# NB_patient_360

Builds `<catalog>.gold.patient_360` (one row per patient) from the silver layer per the STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.patients`, LEFT JOINs to `hospitals`, `insurance_providers` and patient-level aggregated `lab_results`. Full rebuild (CREATE OR REPLACE TABLE AS SELECT).

In [0]:
# <catalog> placeholder from the STTM; override via widget / job parameter
dbutils.widgets.text("catalog", "adb_newcaresync")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print(f"Using catalog: {catalog}")

In [0]:
%sql
CREATE OR REPLACE TABLE gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary (source: silver)'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                             AS total_tests,
    COUNT_IF(result_status = 'Abnormal')                 AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')                 AS critical_test_count,
    MAX(test_date)                                       AS last_test_date
  FROM silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                              AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                          AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                  AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                   AS insurance_provider_id,
  ip.provider_name                                                       AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                        AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                        AS critical_test_count,
  l.last_test_date
FROM silver.patients p
LEFT JOIN silver.hospitals h            ON p.registered_hospital_id = h.hospital_id
LEFT JOIN silver.insurance_providers ip ON p.insurance_provider_id  = ip.insurance_provider_id
LEFT JOIN lab_agg l                     ON p.patient_id             = l.patient_id;

In [0]:
record_count = spark.table("gold.patient_360").count()
dbutils.notebook.exit(str(record_count))